In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, copy, random, time, gc, itertools
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

In [ ]:
# heartbeat utilities

HEARTBEAT_SECS = 600

def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

In [ ]:
# paths


# data + base project
DATASET_PATH  = "/content/drive/MyDrive/298/Jena/Data/jena_climate_2009_2016.csv"
BASE_PROJ_DIR = "/content/drive/MyDrive/298/Jena/proj_v2"

# base model
BASE_MODEL_DIR          = os.path.join(BASE_PROJ_DIR, "baseModel")
BASE_STUDENTS_DIR       = os.path.join(BASE_MODEL_DIR, "baseStudents")
BASE_LOGS_DIR           = os.path.join(BASE_MODEL_DIR, "logs")
BASE_TRAIN_HISTORY_CSV  = os.path.join(BASE_LOGS_DIR, "jena_base_train_history.csv")

# vanilla students
FGSM_DIR                = os.path.join(BASE_PROJ_DIR, "fgsm")
FGSM_ADV_DIR            = os.path.join(FGSM_DIR, "advStudents")
FGSM_RESULTS_DIR        = os.path.join(FGSM_DIR, "results")
FGSM_LOGS_DIR           = os.path.join(FGSM_RESULTS_DIR, "train_logs")

BIM_DIR                 = os.path.join(BASE_PROJ_DIR, "bim")
BIM_ADV_DIR             = os.path.join(BIM_DIR, "advStudents")
BIM_RESULTS_DIR         = os.path.join(BIM_DIR, "results")
BIM_LOGS_DIR            = os.path.join(BIM_RESULTS_DIR, "train_logs")

PGD_DIR                 = os.path.join(BASE_PROJ_DIR, "pgd")
PGD_ADV_DIR             = os.path.join(PGD_DIR, "advStudents")
PGD_RESULTS_DIR         = os.path.join(PGD_DIR, "results")
PGD_LOGS_DIR            = os.path.join(PGD_RESULTS_DIR, "train_logs")

# vanilla pipeline CSVs
FGSM_RUNS_CSV               = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_runs.csv")
FGSM_STATS_CSV              = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_stats.csv")
FGSM_WEIGHT_DIVERSITY_CSV   = os.path.join(FGSM_RESULTS_DIR, "fgsm_weight_diversity.csv")
FGSM_PRED_DIVERSITY_CSV     = os.path.join(FGSM_RESULTS_DIR, "fgsm_prediction_diversity.csv")
FGSM_XFER_MATRIX_CSV        = os.path.join(FGSM_RESULTS_DIR, "fgsm_transfer_matrix.csv")

BIM_RUNS_CSV                = os.path.join(BIM_RESULTS_DIR, "bim_morphence_runs.csv")
BIM_STATS_CSV               = os.path.join(BIM_RESULTS_DIR, "bim_morphence_stats.csv")
BIM_WEIGHT_DIVERSITY_CSV    = os.path.join(BIM_RESULTS_DIR, "bim_weight_diversity.csv")
BIM_PRED_DIVERSITY_CSV      = os.path.join(BIM_RESULTS_DIR, "bim_prediction_diversity.csv")
BIM_XFER_MATRIX_CSV         = os.path.join(BIM_RESULTS_DIR, "bim_transfer_matrix.csv")

PGD_RUNS_CSV                = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_runs.csv")
PGD_STATS_CSV               = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_stats.csv")
PGD_WEIGHT_DIVERSITY_CSV    = os.path.join(PGD_RESULTS_DIR, "pgd_weight_diversity.csv")
PGD_PRED_DIVERSITY_CSV      = os.path.join(PGD_RESULTS_DIR, "pgd_prediction_diversity.csv")
PGD_XFER_MATRIX_CSV         = os.path.join(PGD_RESULTS_DIR, "pgd_transfer_matrix.csv")

# jenaNov paths
JENANOV_DIR         = os.path.join(BASE_PROJ_DIR, "jenaNov")
STUDENTS_NOV_DIR    = os.path.join(JENANOV_DIR, "studentsNov")
JENANOV_LOGS_DIR    = os.path.join(JENANOV_DIR, "logs")

# fgsm nov
FGSM_NOV_DIR                = os.path.join(JENANOV_DIR, "fgsm")
FGSM_NOV_STUDENTS_ADV_DIR   = os.path.join(FGSM_NOV_DIR, "studentsNovAdv")
FGSM_NOV_RESULTS_DIR        = os.path.join(FGSM_NOV_DIR, "results")
FGSM_NOV_LOGS_DIR           = os.path.join(FGSM_NOV_RESULTS_DIR, "train_logs")

NOV_FGSM_RUNS_30_CSV        = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_runs_30.csv")
NOV_FGSM_STATS_30_CSV       = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_stats_30.csv")
NOV_FGSM_WEIGHT_DIVERSITY   = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_weight_diversity.csv")
NOV_FGSM_PRED_DIVERSITY     = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_prediction_diversity.csv")
NOV_FGSM_XFER_MATRIX_CSV    = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_transfer_matrix.csv")

# bim nov
BIM_NOV_DIR                 = os.path.join(JENANOV_DIR, "bim")
BIM_NOV_STUDENTS_ADV_DIR    = os.path.join(BIM_NOV_DIR, "studentsNovAdv")
BIM_NOV_RESULTS_DIR         = os.path.join(BIM_NOV_DIR, "results")
BIM_NOV_LOGS_DIR            = os.path.join(BIM_NOV_RESULTS_DIR, "train_logs")
BIM_NOV_CRASH_DIR           = os.path.join(BIM_NOV_DIR, "crash_dump")

NOV_BIM_RUNS_30_CSV         = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_morphence_runs_30.csv")
NOV_BIM_STATS_30_CSV        = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_morphence_stats_30.csv")
NOV_BIM_WEIGHT_DIVERSITY    = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_weight_diversity.csv")
NOV_BIM_PRED_DIVERSITY      = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_prediction_diversity.csv")
NOV_BIM_XFER_MATRIX_CSV     = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_transfer_matrix.csv")

# pgd nov
PGD_NOV_DIR                 = os.path.join(JENANOV_DIR, "pgd")
PGD_NOV_STUDENTS_ADV_DIR    = os.path.join(PGD_NOV_DIR, "studentsNovAdv")
PGD_NOV_RESULTS_DIR         = os.path.join(PGD_NOV_DIR, "results")
PGD_NOV_LOGS_DIR            = os.path.join(PGD_NOV_RESULTS_DIR, "train_logs")
PGD_NOV_CRASH_DIR           = os.path.join(PGD_NOV_DIR, "crash_dump")

NOV_PGD_RUNS_30_CSV         = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_morphence_runs_30.csv")
NOV_PGD_STATS_30_CSV        = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_morphence_stats_30.csv")
NOV_PGD_WEIGHT_DIVERSITY    = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_weight_diversity.csv")
NOV_PGD_PRED_DIVERSITY      = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_prediction_diversity.csv")
NOV_PGD_XFER_MATRIX_CSV     = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_transfer_matrix.csv")

# create all dirs
for d in [
    BASE_PROJ_DIR, BASE_MODEL_DIR, BASE_STUDENTS_DIR, BASE_LOGS_DIR,
    FGSM_DIR, FGSM_ADV_DIR, FGSM_RESULTS_DIR, FGSM_LOGS_DIR,
    BIM_DIR,  BIM_ADV_DIR,  BIM_RESULTS_DIR,  BIM_LOGS_DIR,
    PGD_DIR,  PGD_ADV_DIR,  PGD_RESULTS_DIR,  PGD_LOGS_DIR,
    JENANOV_DIR, STUDENTS_NOV_DIR, JENANOV_LOGS_DIR,
    FGSM_NOV_DIR, FGSM_NOV_STUDENTS_ADV_DIR, FGSM_NOV_RESULTS_DIR, FGSM_NOV_LOGS_DIR,
    BIM_NOV_DIR,  BIM_NOV_STUDENTS_ADV_DIR,  BIM_NOV_RESULTS_DIR,  BIM_NOV_LOGS_DIR, BIM_NOV_CRASH_DIR,
    PGD_NOV_DIR,  PGD_NOV_STUDENTS_ADV_DIR,  PGD_NOV_RESULTS_DIR,  PGD_NOV_LOGS_DIR, PGD_NOV_CRASH_DIR,
]:
    os.makedirs(d, exist_ok=True)

print("BASE_PROJ_DIR :", BASE_PROJ_DIR)
print("BASE_MODEL_DIR:", BASE_MODEL_DIR)
print("JENANOV_DIR   :", JENANOV_DIR)
print("STUDENTS_NOV  :", STUDENTS_NOV_DIR)
print("FGSM_NOV_DIR  :", FGSM_NOV_DIR)
print("BIM_NOV_DIR   :", BIM_NOV_DIR)
print("PGD_NOV_DIR   :", PGD_NOV_DIR)


BASE_PROJ_DIR : /content/drive/MyDrive/298/Jena/proj_v2
BASE_MODEL_DIR: /content/drive/MyDrive/298/Jena/proj_v2/baseModel
JENANOV_DIR   : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov
STUDENTS_NOV  : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov
FGSM_NOV_DIR  : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm
BIM_NOV_DIR   : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim
PGD_NOV_DIR   : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd


In [ ]:
# global configs

import threading

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

LOOKBACK   = 24
TARGET_COL = "t_degc"
BATCH      = 64
N_STUDENTS = 4

EPS_LIST = [0.1, 0.2, 0.3, 0.5]
FGSM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
BIM_TRAIN_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
PGD_TRAIN_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]

BIM_ITERS  = 10
PGD_ITERS  = 10

FGSM_EPOCHS = 5
BIM_EPOCHS  = 5
PGD_EPOCHS  = 5

LR_BASE    = 1e-3
LR_STUDENT = 1e-3

Device: cuda


In [ ]:
# data prep

df = pd.read_csv(DATASET_PATH)
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df = df.set_index("Date Time").sort_index()
df = df[~df.index.duplicated(keep="first")]

df = df.resample("h").ffill()
if df.head(1).isna().any(axis=None):
    df = df.bfill(limit=1)

col_map = {
    "p (mbar)": "p_mbar",
    "T (degC)": "t_degc",
    "Tpot (K)": "tpot_k",
    "rh (%)":   "rh_pct",
    "wv (m/s)": "wv_ms",
}
df_sel = df[list(col_map.keys())].rename(columns=col_map)

print("After hourly resample + ffill/bfill:")
print("Shape:", df_sel.shape)
print("Columns:", df_sel.columns.tolist())

split_idx = int(0.8 * len(df_sel))
train_df, test_df = df_sel.iloc[:split_idx].copy(), df_sel.iloc[split_idx:].copy()

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(train_df),
    index=train_df.index,
    columns=train_df.columns,
)
test_scaled = pd.DataFrame(
    scaler.transform(test_df),
    index=test_df.index,
    columns=test_df.columns,
)

def make_windows(frame, lookback=24, target_col="t_degc"):
    vals = frame.values
    tgt_idx = frame.columns.get_loc(target_col)
    X, y, idx = [], [], []
    for i in range(lookback, len(vals)):
        X.append(vals[i-lookback:i])
        y.append(vals[i][tgt_idx])
        idx.append(frame.index[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(idx)

X_train, y_train, t_train = make_windows(train_scaled, LOOKBACK, TARGET_COL)
X_test,  y_test,  t_test  = make_windows(test_scaled,  LOOKBACK, TARGET_COL)

print("\nWindowed tensors")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

N_tr = len(X_train)
n_tr_core = int(0.9 * N_tr)

X_tr_core, y_tr_core = X_train[:n_tr_core], y_train[:n_tr_core]
X_val,     y_val     = X_train[n_tr_core:], y_train[n_tr_core:]

print("\nTrain/Val/Test windowed splits:")
print("  train_core:", X_tr_core.shape, y_tr_core.shape)
print("  val       :", X_val.shape,     y_val.shape)
print("  test      :", X_test.shape,    y_test.shape)

class TSForecastDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_dl = DataLoader(
    TSForecastDataset(X_tr_core, y_tr_core),
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
)

val_dl = DataLoader(
    TSForecastDataset(X_val, y_val),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
)

test_dl = DataLoader(
    TSForecastDataset(X_test, y_test),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
)

After hourly resample + ffill/bfill:
Shape: (70129, 5)
Columns: ['p_mbar', 't_degc', 'tpot_k', 'rh_pct', 'wv_ms']

Windowed tensors
X_train: (56079, 24, 5) y_train: (56079,)
X_test:  (14002, 24, 5) y_test:  (14002,)

Train/Val/Test windowed splits:
  train_core: (50471, 24, 5) (50471,)
  val       : (5608, 24, 5) (5608,)
  test      : (14002, 24, 5) (14002,)


In [ ]:
# model

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerRegressor(nn.Module):
    def __init__(self, in_feats=5, d_model=128, nhead=4,
                 num_layers=4, dim_ff=256, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(in_feats, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.posenc  = PositionalEncoding(d_model)
        self.head    = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        z  = self.in_proj(x)
        z  = self.posenc(z)
        z  = self.encoder(z)
        zT = z[:, -1, :]
        return self.head(zT)

mse = nn.MSELoss()
def rmse_loss(pred, true): return torch.sqrt(mse(pred, true))

mse_loss = mse

def quantile_loss(true, pred, q=0.5):
    e = true - pred
    return torch.mean(torch.maximum(q*e, (q-1)*e))

In [ ]:
# load base model

base_best_ckpt = os.path.join(BASE_MODEL_DIR, "jena_base_best.pth")
assert os.path.exists(base_best_ckpt), f"Base checkpoint not found: {base_best_ckpt}"
print("Using base checkpoint:", base_best_ckpt)

# load to gpu for eval + attacks
base = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
base.load_state_dict(torch.load(base_best_ckpt, map_location=device))
base.eval()
print("Base model loaded on device:", device)

# load to cpu for student generation
base_cpu = TransformerRegressor(in_feats=X_train.shape[-1]).to("cpu")
base_cpu.load_state_dict(torch.load(base_best_ckpt, map_location="cpu"))
base_cpu.eval()
print("Base model loaded on CPU for student generation.")

base_eval = base

Using base checkpoint: /content/drive/MyDrive/298/Jena/proj_v2/baseModel/jena_base_best.pth
Base model loaded on device: cuda
Base model loaded on CPU for student generation.


In [ ]:
# student model size logging

def model_n_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters())

def model_file_size_mb(path: str):
    if not os.path.exists(path):
        return None
    return os.path.getsize(path) / (1024.0 * 1024.0)

def append_df_to_csv(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

MODEL_SIZES_NOV_CSV = os.path.join(JENANOV_DIR, "jenaNov_model_sizes.csv")

In [ ]:
# student generation parameters

def is_enc_self_attn_param(name: str) -> bool:
    return name.startswith("encoder.layers.") and ".self_attn." in name

def is_ffn_param(name: str) -> bool:
    if not name.startswith("encoder.layers."):
        return False
    return (".linear1." in name) or (".linear2." in name)

def is_norm_param(name: str) -> bool:
    return name.startswith("encoder.layers.") and ".norm" in name

def is_inproj_head_param(name: str) -> bool:
    return name.startswith("in_proj") or name.startswith("head")

novel_student_specs = [
    dict(
        idx=5,
        name="student_05_enc_attn",
        selector=is_enc_self_attn_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=6,
        name="student_06_ffn",
        selector=is_ffn_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=7,
        name="student_07_norms",
        selector=is_norm_param,
        noise_scale=1e-3,
    ),
    dict(
        idx=8,
        name="student_08_inproj_head",
        selector=is_inproj_head_param,
        noise_scale=1e-3,
    ),
]

In [ ]:

# novel student parameter groups
#
# student 05  -  encoder self attention weights only
#                includes:
#                   encoder.layers.*.self_attn.in_proj_weight
#                   encoder.layers.*.self_attn.in_proj_bias
#                   encoder.layers.*.self_attn.out_proj.weight
#                   encoder.layers.*.self_attn.out_proj.bias
#
# student 06  -  encoder feed forward layers (MLP) only
#                includes:
#                   encoder.layers.*.linear1.weight / bias
#                   encoder.layers.*.linear2.weight / bias
#
# student 07  -  encoder LayerNorms only
#                includes:
#                   encoder.layers.*.norm1.weight / bias
#                   encoder.layers.*.norm2.weight / bias
#
# student 08  -  input projection + prediction head only
#                includes:
#                   in_proj.weight / bias
#                   head.(0).weight  (LayerNorm)
#                   head.(0).bias
#                   head.(1).weight  (final linear regressor)
#                   head.(1).bias
#
#
# each student receives perturbations ONLY in the subset above
#
# all remaining model parameters stay identical to the base model

In [ ]:
# student generation

novel_student_paths = []

print("\n[JenaNov] creating novel students from base_cpu")

for spec in novel_student_specs:
    idx      = spec["idx"]
    name     = spec["name"]
    selector = spec["selector"]
    sigma    = spec["noise_scale"]

    print(f"  [Novel Student {idx}] {name} | sigma={sigma}")

    # copy the trained base CPU
    st = copy.deepcopy(base_cpu)

    # perturb only selected parameters
    with torch.no_grad():
        for pname, p in st.named_parameters():
            if selector(pname):
                p.add_(sigma * torch.randn_like(p))

    # save to disk
    out_path = os.path.join(STUDENTS_NOV_DIR, f"{name}.pth")
    torch.save(st.state_dict(), out_path)
    novel_student_paths.append((idx, name, out_path))
    print("    saved to:", out_path)

    # log file size
    size_mb = model_file_size_mb(out_path)
    df_ms = pd.DataFrame([{
        "student_idx": idx,
        "model_name": name,
        "path": out_path,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_NOV_CSV)

# sort by index
novel_student_paths = sorted(novel_student_paths, key=lambda x: x[0])

print("\n[JenaNov] novel students generated:")
for idx, name, path in novel_student_paths:
    print(f"  {idx}: {name} - {path}")


[JenaNov] creating novel students from base_cpu
  [Novel Student 5] student_05_enc_attn | sigma=0.001
    saved to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_05_enc_attn.pth
  [Novel Student 6] student_06_ffn | sigma=0.001
    saved to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_06_ffn.pth
  [Novel Student 7] student_07_norms | sigma=0.001
    saved to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_07_norms.pth
  [Novel Student 8] student_08_inproj_head | sigma=0.001
    saved to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_08_inproj_head.pth

[JenaNov] novel students generated:
  5: student_05_enc_attn - /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_05_enc_attn.pth
  6: student_06_ffn - /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_06_ffn.pth
  7: student_07_norms - /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_07_norms.pth

In [ ]:
# attacks

#fgsm
def rfgsm(model, X, Y, eps=0.1, alpha=0.02):
    was_training = model.training
    model.eval()

    X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    X_adv.requires_grad_(True)

    loss = mse(model(X_adv), Y)
    loss.backward()

    X_adv = (X_adv + alpha * X_adv.grad.sign()).clamp(0, 1)

    if was_training:
        model.train()
    return X_adv.detach()

#bim
def bim_attack(model, X, Y, eps=0.15, alpha=None, iters=10, random_start=False):
    if alpha is None:
        alpha = eps / max(1, iters)

    was_training = model.training
    model.eval()

    if random_start:
        X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    else:
        X_adv = X.clone().detach()

    X_adv.requires_grad_(True)

    for _ in range(iters):
        preds = model(X_adv)
        loss  = mse(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()
        loss.backward()

        grad_sign = X_adv.grad.sign()
        X_adv     = (X_adv + alpha * grad_sign).detach()

        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0, 1).detach()
        X_adv.requires_grad_(True)

    if was_training:
        model.train()
    return X_adv.detach()

#pgd
def pgd_attack(model, X, Y, eps=0.15, alpha=None, iters=10, random_start=True):
    if alpha is None:
        alpha = eps / max(1, iters)

    was_training = model.training
    model.eval()

    if random_start:
        X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    else:
        X_adv = X.clone().detach()

    X_adv.requires_grad_(True)

    for _ in range(iters):
        preds = model(X_adv)
        loss  = mse(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()
        loss.backward()

        grad_sign = X_adv.grad.sign()
        X_adv     = (X_adv + alpha * grad_sign).detach()

        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0, 1).detach()
        X_adv.requires_grad_(True)

    if was_training:
        model.train()
    return X_adv.detach()


In [ ]:
# random start configs for attacks

BIM_RANDOM_START_TRAIN = False
PGD_RANDOM_START_TRAIN = True
BIM_RANDOM_START_EVAL = True
PGD_RANDOM_START_EVAL = True


In [ ]:
#student adv training definitions

#fgsm
def adv_train_student_fgsm(
    student_path,
    out_dir,
    logs_dir,
    out_suffix="_fgsm_adv",
    epochs=FGSM_EPOCHS,
    lr=LR_STUDENT,
    eps_list=None,
    alpha=0.02,
    q=0.5,
):

    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = FGSM_TRAIN_EPS_LIST

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"fgsm_adv_train_{student_stub}")
    start_time = time.time()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                # clean loss
                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                # multi ε fgsm
                for eps in eps_list:
                    xa = rfgsm(model, xb, yb, eps=eps, alpha=alpha)
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean    += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins    = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch   = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[FGSM NOV train {student_stub}] "
                f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} | ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            # per epoch checkpoint
            ep_ckpt = os.path.join(
                out_dir,
                f"{student_stub}{out_suffix}_epoch{ep:02d}.pth"
            )
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # final fgsm adv nov student
    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth")
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved FGSM-NOV adv student:", out)

    # save train history
    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved FGSM-NOV train history to:", hist_path)

    return out

In [ ]:
# bim adv training

def adv_train_student_bim(
    student_path,
    out_dir,
    logs_dir,
    out_suffix="_bim_adv",
    epochs=5,
    lr=1e-3,
    eps_list=None,
    iters=10,
    q=0.5,
):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = BIM_TRAIN_EPS_LIST

    # BIM uses the pipeline's setting (False)
    random_start = BIM_RANDOM_START_TRAIN

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"bim_adv_train_{student_stub}")
    start_time = time.time()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = bim_attack(
                        model,
                        xb,
                        yb,
                        eps=eps,
                        alpha=eps / max(1, iters),
                        iters=iters,
                        random_start=random_start,
                    )
                    model.train()
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)

                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                La_mean = torch.stack(adv_losses).mean()

                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0

            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[BIM train {student_stub}] epoch {ep}/{epochs} "
                f"| clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} "
                f"| ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            ep_ckpt = os.path.join(
                out_dir,
                f"{student_stub}{out_suffix}_epoch{ep:02d}.pth",
            )
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # save final adv student
    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth"),
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved BIM adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved BIM train history to:", hist_path)

    return out

In [ ]:
# pgd adv training

def adv_train_student_pgd(
    student_path,
    out_dir,
    logs_dir,
    out_suffix="_pgd_adv",
    epochs=5,
    lr=1e-3,
    eps_list=None,
    iters=10,
    q=0.5,
):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = PGD_TRAIN_EPS_LIST

    random_start = PGD_RANDOM_START_TRAIN

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"pgd_adv_train_{student_stub}")
    start_time = time.time()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = pgd_attack(
                        model,
                        xb,
                        yb,
                        eps=eps,
                        alpha=eps / max(1, iters),
                        iters=iters,
                        random_start=random_start,
                    )
                    model.train()
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)

                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                La_mean = torch.stack(adv_losses).mean()

                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0

            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[PGD train {student_stub}] epoch {ep}/{epochs} "
                f"| clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} "
                f"| ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            ep_ckpt = os.path.join(
                out_dir,
                f"{student_stub}{out_suffix}_epoch{ep:02d}.pth",
            )
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # save final adv student
    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth"),
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved PGD adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved PGD train history to:", hist_path)

    return out

In [ ]:
FGSM_NOV_ADV_DIR = FGSM_NOV_STUDENTS_ADV_DIR
BIM_NOV_ADV_DIR  = BIM_NOV_STUDENTS_ADV_DIR
PGD_NOV_ADV_DIR  = PGD_NOV_STUDENTS_ADV_DIR

In [ ]:
# adv training for novel students

print("\n[JenaNov] Training FGSM adv NOV students")
nov_fgsm_adv_paths = []
for idx, name, sp in novel_student_paths:
    out = adv_train_student_fgsm(
        sp,
        out_dir=FGSM_NOV_ADV_DIR,
        logs_dir=FGSM_NOV_LOGS_DIR,
        out_suffix="_fgsm_adv",
        epochs=FGSM_EPOCHS,
        lr=LR_STUDENT,
        eps_list=FGSM_TRAIN_EPS_LIST,
        alpha=0.02,
    )
    nov_fgsm_adv_paths.append((idx, name, out))


In [ ]:
print("\n[JenaNov] Training BIM adv NOV students")
nov_bim_adv_paths = []
for idx, name, sp in novel_student_paths:
    out = adv_train_student_bim(
        sp,
        out_dir=BIM_NOV_ADV_DIR,
        logs_dir=BIM_NOV_LOGS_DIR,
        out_suffix="_bim_adv",
        epochs=BIM_EPOCHS,
        lr=LR_STUDENT,
        eps_list=BIM_TRAIN_EPS_LIST,
        iters=BIM_ITERS,
    )
    nov_bim_adv_paths.append((idx, name, out))


[JenaNov] Training BIM adv NOV students
[BIM train student_05_enc_attn] epoch 1/5 | clean=0.2068 | adv_mean=0.2104 | ep=5.90m | total=5.90m
[heartbeat:bim_adv_train_student_05_enc_attn] running... elapsed=10.00 min
[BIM train student_05_enc_attn] epoch 2/5 | clean=0.2084 | adv_mean=0.2086 | ep=5.88m | total=11.78m
[BIM train student_05_enc_attn] epoch 3/5 | clean=0.2081 | adv_mean=0.2082 | ep=5.97m | total=17.75m
[heartbeat:bim_adv_train_student_05_enc_attn] running... elapsed=20.00 min
[BIM train student_05_enc_attn] epoch 4/5 | clean=0.2082 | adv_mean=0.2082 | ep=5.91m | total=23.67m
[BIM train student_05_enc_attn] epoch 5/5 | clean=0.2082 | adv_mean=0.2082 | ep=5.90m | total=29.57m
Saved BIM adv student: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/studentsNovAdv/student_05_enc_attn_bim_adv.pth
Saved BIM train history to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/results/train_logs/student_05_enc_attn_bim_adv_train_history.csv
[BIM train student_06_ffn] epoch 1/5 |

In [ ]:
print("\n[JenaNov] Training PGD adv NOV students")
nov_pgd_adv_paths = []
for idx, name, sp in novel_student_paths:
    out = adv_train_student_pgd(
        sp,
        out_dir=PGD_NOV_ADV_DIR,
        logs_dir=PGD_NOV_LOGS_DIR,
        out_suffix="_pgd_adv",
        epochs=PGD_EPOCHS,
        lr=LR_STUDENT,
        eps_list=PGD_TRAIN_EPS_LIST,
        iters=PGD_ITERS,
    )
    nov_pgd_adv_paths.append((idx, name, out))


[JenaNov] Training PGD adv NOV students
[PGD train student_05_enc_attn] epoch 1/5 | clean=0.1208 | adv_mean=0.2095 | ep=5.86m | total=5.86m
[heartbeat:pgd_adv_train_student_05_enc_attn] running... elapsed=10.00 min
[PGD train student_05_enc_attn] epoch 2/5 | clean=0.0710 | adv_mean=0.1748 | ep=5.90m | total=11.76m
[PGD train student_05_enc_attn] epoch 3/5 | clean=0.0568 | adv_mean=0.1724 | ep=5.89m | total=17.65m
[heartbeat:pgd_adv_train_student_05_enc_attn] running... elapsed=20.00 min
[PGD train student_05_enc_attn] epoch 4/5 | clean=0.0481 | adv_mean=0.1769 | ep=5.87m | total=23.52m
[PGD train student_05_enc_attn] epoch 5/5 | clean=0.0478 | adv_mean=0.1751 | ep=5.89m | total=29.41m
Saved PGD adv student: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd/studentsNovAdv/student_05_enc_attn_pgd_adv.pth
Saved PGD train history to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd/results/train_logs/student_05_enc_attn_pgd_adv_train_history.csv
[PGD train student_06_ffn] epoch 1/5 |

In [ ]:
print("\n[JenaNov] Done adversarial training for NOV students.")
print("  FGSM adv paths:", [p for (_, _, p) in nov_fgsm_adv_paths])
print("  BIM  adv paths:", [p for (_, _, p) in nov_bim_adv_paths])
print("  PGD  adv paths:", [p for (_, _, p) in nov_pgd_adv_paths])


[JenaNov] Done adversarial training for NOV students.
  FGSM adv paths: ['/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/studentsNovAdv/student_05_enc_attn_fgsm_adv.pth', '/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/studentsNovAdv/student_06_ffn_fgsm_adv.pth', '/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/studentsNovAdv/student_07_norms_fgsm_adv.pth', '/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/studentsNovAdv/student_08_inproj_head_fgsm_adv.pth']
  BIM  adv paths: ['/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/studentsNovAdv/student_05_enc_attn_bim_adv.pth', '/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/studentsNovAdv/student_06_ffn_bim_adv.pth', '/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/studentsNovAdv/student_07_norms_bim_adv.pth', '/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/studentsNovAdv/student_08_inproj_head_bim_adv.pth']
  PGD  adv paths: ['/content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd/studentsNovAdv/studen

In [ ]:
# eval helpers

def eval_clean_rmse(model, loader):
    """
    RMSE of a single model on clean (non-adversarial) data.
    """
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            p  = model(xb)
            ys.append(yb.cpu().numpy())
            ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))


def eval_ensemble_rmse(models, loader):
    """
    RMSE of an ensemble: average predictions of all models, then compute RMSE.
    """
    for m in models:
        m.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = []
            for m in models:
                preds.append(m(xb))
            preds = torch.stack(preds, dim=0).mean(dim=0)
            ys.append(yb.cpu().numpy())
            ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))


def eval_pair_attack(defender, attacker, loader, attack_fn, atk_kwargs):
    """
    RMSE when 'attacker' crafts adversarial examples using attack_fn,
    and 'defender' is evaluated on those adversarial inputs.
    attack_fn must have signature: attack_fn(model, X, Y, **atk_kwargs)
    """
    defender.eval()
    attacker.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        xa = attack_fn(attacker, xb, yb, **atk_kwargs)
        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))


def eval_random_switch_attack(models, loader, attack_fn, atk_kwargs_func):
    """
    Morphence-style random switch:
      - For each batch, pick a random attacker and defender from 'models'.
      - atk_kwargs_func() returns a dict with eps/alpha/iters/random_start, etc.
      - attack_fn is one of: rfgsm_attack, bim_attack, pgd_attack
        with signature attack_fn(model, X, Y, **atk_kwargs).
    """
    for m in models:
        m.eval()
    ys, ps = [], []
    n_stu = len(models)

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        atk_model = random.choice(models)
        def_model = random.choice(models)

        atk_kwargs = atk_kwargs_func()

        xa = attack_fn(atk_model, xb, yb, **atk_kwargs)
        with torch.no_grad():
            p = def_model(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))


In [ ]:
# diversity metrics
# weight + prediction

def flatten_params(model: nn.Module):
    """
    Flatten all parameters of a model into a single 1D vector (on CPU).
    Used for weight-space distance calculations.
    """
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])


def compute_weight_diversity(students, names, out_csv):
    """
    Compute pairwise L2 distances between all student parameter vectors.

    Args:
      students: list[nn.Module]  (e.g., novel FGSM/BIM/PGD students)
      names:    list[str]        (same length as `students`)
      out_csv:  path to save CSV with columns:
                  [student_i, student_j, l2_distance]
    """
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i, j in itertools.combinations(range(k), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(out_csv, index=False)
    print("Saved weight diversity to:", out_csv)
    return df_w


@torch.no_grad()
def compute_prediction_diversity(students, loader, out_csv):
    """
    Compute prediction diversity across students:
      - For each batch, each student predicts y_hat (B,1) -> flatten to (B,)
      - Average over batch dimension to get one scalar per student
      - Accumulate mean and mean-square across batches
      - Variance per student is mean(y)^2 adjustment

    CSV has columns: [student_idx, mean_pred, var_pred]
    """
    for m in students:
        m.eval()

    k = len(students)
    sum_pred  = None
    sum_pred2 = None
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)

        preds_stu = []
        for m in students:
            # each: (B, 1) -> flatten to (B,)
            p = m(xb).cpu().numpy().reshape(-1)
            preds_stu.append(p)

        # (k, B)
        preds_stu = np.stack(preds_stu, axis=0)

        # average over batch dimension -> (k,)
        preds_batch_mean = preds_stu.mean(axis=1)

        if sum_pred is None:
            sum_pred  = np.zeros_like(preds_batch_mean, dtype=np.float64)
            sum_pred2 = np.zeros_like(preds_batch_mean, dtype=np.float64)

        sum_pred  += preds_batch_mean
        sum_pred2 += preds_batch_mean ** 2
        count     += 1

    if count == 0:
        raise RuntimeError("compute_prediction_diversity: loader produced 0 batches")

    mean_pred  = sum_pred  / count          # (k,)
    mean_pred2 = sum_pred2 / count          # (k,)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "student_idx": np.arange(k),
        "mean_pred":   mean_pred,
        "var_pred":    var_pred,
    })
    df_p.to_csv(out_csv, index=False)
    print("Saved prediction diversity to:", out_csv)
    return df_p

In [ ]:
# transferability matrix
# per epsilon, per attack type

def compute_transfer_matrix(
    students,
    names,
    loader,
    attack_fn,
    eps_list,
    extra_kwargs_func,
    out_csv,
    attack_label="rfgsm",
):
    """
    Compute a transfer matrix:
      - rows: attacker model
      - cols: defender model
      - entries: RMSE(defender(attack_fn(attacker, X, Y, eps,...)))

    Args:
      students:  list[nn.Module] (e.g., FGSM/BIM/PGD NOV students)
      names:     list[str], same length as students
      loader:    DataLoader over (X, Y) pairs
      attack_fn: function(model, X, Y, **atk_kwargs) -> X_adv
      eps_list:  list[float], epsilons to evaluate
      extra_kwargs_func: callable(eps) -> dict of kwargs
                         (e.g., {"eps": eps, "alpha": eps/iters, "iters": 10})
      out_csv:   path to store CSV
      attack_label: string label for "attack" column ("rfgsm", "bim", "pgd")
    """
    rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_label}_transfer_matrix")
    start_time = time.time()

    try:
        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)
            print(f"\n[{attack_label.upper()}] Transfer matrix for eps={eps}...")
            for i, atk_name in enumerate(names):
                for j, def_name in enumerate(names):
                    atk_model = students[i]
                    def_model = students[j]
                    rmse_ij = eval_pair_attack(
                        defender=def_model,
                        attacker=atk_model,
                        loader=loader,
                        attack_fn=attack_fn,
                        atk_kwargs=atk_kwargs,
                    )
                    rows.append({
                        "attack": attack_label,
                        "epsilon": eps,
                        "attacker": atk_name,
                        "defender": def_name,
                        "rmse_adv": rmse_ij,
                    })
                    print(
                        f"  atk={atk_name} - def={def_name} | "
                        f"eps={eps:.2f} | RMSE={rmse_ij:.5f}"
                    )
    finally:
        stop_heartbeat(hb_flag, hb_thread)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {attack_label} transfer matrix to:", out_csv)
    return df

In [ ]:
# reload nov base students + nov adversarial students


# nov base students
nov_student_paths = sorted([
    os.path.join(STUDENTS_NOV_DIR, f)
    for f in os.listdir(STUDENTS_NOV_DIR)
    if f.endswith(".pth")
])

print("NOV base students found:", len(nov_student_paths))
for p in nov_student_paths:
    print("  ", p)

# FGSM nov adv students
fgsm_nov_adv_paths = sorted([
    os.path.join(FGSM_NOV_STUDENTS_ADV_DIR, f)
    for f in os.listdir(FGSM_NOV_STUDENTS_ADV_DIR)
    if f.endswith("_fgsm_adv.pth")
])

print("\nFGSM NOV adv students found:", len(fgsm_nov_adv_paths))
for p in fgsm_nov_adv_paths:
    print("  ", p)

# BIM nov adv students
bim_nov_adv_paths = sorted([
    os.path.join(BIM_NOV_STUDENTS_ADV_DIR, f)
    for f in os.listdir(BIM_NOV_STUDENTS_ADV_DIR)
    if f.endswith("_bim_adv.pth")
])

print("\nBIM NOV adv students found:", len(bim_nov_adv_paths))
for p in bim_nov_adv_paths:
    print("  ", p)

# PGD nov adv students
pgd_nov_adv_paths = sorted([
    os.path.join(PGD_NOV_STUDENTS_ADV_DIR, f)
    for f in os.listdir(PGD_NOV_STUDENTS_ADV_DIR)
    if f.endswith("_pgd_adv.pth")
])

print("\nPGD NOV adv students found:", len(pgd_nov_adv_paths))
for p in pgd_nov_adv_paths:
    print("  ", p)


# move nov adv students to gpu
def load_nov_adv_students(path_list):
    students = []
    for p in path_list:
        m = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
        m.load_state_dict(torch.load(p, map_location=device))
        m.eval()
        students.append(m)
    return students

fgsm_nov_students = load_nov_adv_students(fgsm_nov_adv_paths)
bim_nov_students  = load_nov_adv_students(bim_nov_adv_paths)
pgd_nov_students  = load_nov_adv_students(pgd_nov_adv_paths)

fgsm_nov_student_names = [os.path.basename(p).replace(".pth", "") for p in fgsm_nov_adv_paths]
bim_nov_student_names  = [os.path.basename(p).replace(".pth", "") for p in bim_nov_adv_paths]
pgd_nov_student_names  = [os.path.basename(p).replace(".pth", "") for p in pgd_nov_adv_paths]

print("\nLoaded NOV adv student pools:")
print(" FGSM NOV:", len(fgsm_nov_students))
print(" BIM  NOV:", len(bim_nov_students))
print(" PGD  NOV:", len(pgd_nov_students))


NOV base students found: 4
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_05_enc_attn.pth
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_06_ffn.pth
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_07_norms.pth
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/studentsNov/student_08_inproj_head.pth

FGSM NOV adv students found: 4
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/studentsNovAdv/student_05_enc_attn_fgsm_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/studentsNovAdv/student_06_ffn_fgsm_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/studentsNovAdv/student_07_norms_fgsm_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/studentsNovAdv/student_08_inproj_head_fgsm_adv.pth

BIM NOV adv students found: 4
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/studentsNovAdv/student_05_enc_attn_bim_adv.pth
   /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim

In [ ]:
# 30run eval

def run_morphence_eval(
    attack_name,
    attack_fn,
    eps_list,
    students,
    results_csv,
    stats_csv,
    extra_kwargs_func,
    n_runs=30,
):
    """
    attack_name: "rfgsm", "bim", "pgd"
    attack_fn:   rfgsm, bim_attack, pgd_attack
    eps_list:    list of epsilons
    students:    list of adv-trained student models (NOV pool in this notebook)
    extra_kwargs_func: function(eps) -> dict of kwargs to pass to attack_fn
    """
    all_rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_name}_morphence_eval")
    start_all = time.time()

    try:
        for run_i in range(1, n_runs + 1):
            run_start = time.time()
            seed_i = 3000 + run_i
            random.seed(seed_i)
            np.random.seed(seed_i)
            torch.manual_seed(seed_i)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed_i)

            # clean
            base_clean  = eval_clean_rmse(base_eval, test_dl)
            morph_clean = eval_ensemble_rmse(students, test_dl)
            all_rows.append(["clean", None, "base",           base_clean,  run_i, seed_i])
            all_rows.append(["clean", None, "morph_ensemble", morph_clean, run_i, seed_i])

            print(f"[{attack_name.upper()} | Run {run_i:02d}] clean | "
                  f"base={base_clean:.5f} | morph_ens={morph_clean:.5f}")

            # attack
            for eps in eps_list:
                atk_kwargs = extra_kwargs_func(eps)

                # base white box (attacker = defender = base_eval)
                base_rmse = eval_pair_attack(
                    defender=base_eval,
                    attacker=base_eval,
                    loader=test_dl,
                    attack_fn=attack_fn,
                    atk_kwargs=atk_kwargs,
                )

                # random switching within nov pool
                def atk_kwargs_wrapper():
                    return atk_kwargs

                morph_rmse = eval_random_switch_attack(
                    models=students,
                    loader=test_dl,
                    attack_fn=attack_fn,
                    atk_kwargs_func=atk_kwargs_wrapper,
                )

                all_rows.append([attack_name, eps, "base",       base_rmse,  run_i, seed_i])
                all_rows.append([attack_name, eps, "morph_rand", morph_rmse, run_i, seed_i])

                print(
                    f"[{attack_name.upper()} | Run {run_i:02d}] eps={eps:.2f} | "
                    f"base={base_rmse:.5f} | morph_rand={morph_rmse:.5f}"
                )

            # incremental save
            df_runs = pd.DataFrame(
                all_rows,
                columns=["attack","epsilon","model","RMSE","run","seed"]
            )
            df_runs.to_csv(results_csv, index=False)

            stats = (
                df_runs
                .groupby(["attack","epsilon","model"])["RMSE"]
                .agg(mean="mean", std="std", n="count")
                .reset_index()
                .sort_values(["attack","epsilon","model"])
            )
            stats.to_csv(stats_csv, index=False)

            run_mins   = (time.time() - run_start) / 60.0
            total_mins = (time.time() - start_all) / 60.0
            print(
                f"[{attack_name.upper()}] Completed run {run_i}/{n_runs} | "
                f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
                flush=True
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
            gc.collect()

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    print(f"\nSaved {attack_name} runs to:", results_csv)
    print(f"Saved {attack_name} stats to:", stats_csv)
    return df_runs, stats


In [ ]:
print("\n[FGSM-Nov] diversity + transferability + 30-run eval")

def fgsm_nov_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps,
    }

_ = compute_weight_diversity(
    fgsm_nov_students,
    fgsm_nov_student_names,
    NOV_FGSM_WEIGHT_DIVERSITY,
)
_ = compute_prediction_diversity(
    fgsm_nov_students,
    test_dl,
    NOV_FGSM_PRED_DIVERSITY,
)

_ = compute_transfer_matrix(
    students=fgsm_nov_students,
    names=fgsm_nov_student_names,
    loader=test_dl,
    attack_fn=rfgsm,
    eps_list=EPS_LIST,
    extra_kwargs_func=fgsm_nov_kwargs,
    out_csv=NOV_FGSM_XFER_MATRIX_CSV,
    attack_label="rfgsm",         # keep label same as vanilla
)

df_fgsm_nov_runs, fgsm_nov_stats = run_morphence_eval(
    attack_name="rfgsm",          # keep same label for comparison
    attack_fn=rfgsm,
    eps_list=EPS_LIST,
    students=fgsm_nov_students,
    results_csv=NOV_FGSM_RUNS_30_CSV,
    stats_csv=NOV_FGSM_STATS_30_CSV,
    extra_kwargs_func=fgsm_nov_kwargs,
    n_runs=30,
)


[FGSM-Nov] diversity + transferability + 30-run eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/results/nov_fgsm_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/results/nov_fgsm_prediction_diversity.csv

[RFGSM] Transfer matrix for eps=0.1...
  atk=student_05_enc_attn_fgsm_adv - def=student_05_enc_attn_fgsm_adv | eps=0.10 | RMSE=0.11120
  atk=student_05_enc_attn_fgsm_adv - def=student_06_ffn_fgsm_adv | eps=0.10 | RMSE=0.10661
  atk=student_05_enc_attn_fgsm_adv - def=student_07_norms_fgsm_adv | eps=0.10 | RMSE=0.10823
  atk=student_05_enc_attn_fgsm_adv - def=student_08_inproj_head_fgsm_adv | eps=0.10 | RMSE=0.10332
  atk=student_06_ffn_fgsm_adv - def=student_05_enc_attn_fgsm_adv | eps=0.10 | RMSE=0.10956
  atk=student_06_ffn_fgsm_adv - def=student_06_ffn_fgsm_adv | eps=0.10 | RMSE=0.10882
  atk=student_06_ffn_fgsm_adv - def=student_07_norms_fgsm_adv | eps=0.10 | RMSE=0.10858
  atk=student_06_f

In [ ]:
print("\n[BIM-Nov] diversity + transferability + 30-run eval")

def bim_nov_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / BIM_ITERS,
        "iters": BIM_ITERS,
        "random_start": True,
    }

_ = compute_weight_diversity(
    bim_nov_students,
    bim_nov_student_names,
    NOV_BIM_WEIGHT_DIVERSITY,
)
_ = compute_prediction_diversity(
    bim_nov_students,
    test_dl,
    NOV_BIM_PRED_DIVERSITY,
)

_ = compute_transfer_matrix(
    students=bim_nov_students,
    names=bim_nov_student_names,
    loader=test_dl,
    attack_fn=bim_attack,
    eps_list=EPS_LIST,
    extra_kwargs_func=bim_nov_kwargs,
    out_csv=NOV_BIM_XFER_MATRIX_CSV,
    attack_label="bim_nov",
)

df_bim_nov_runs, bim_nov_stats = run_morphence_eval(
    attack_name="bim_nov",
    attack_fn=bim_attack,
    eps_list=EPS_LIST,
    students=bim_nov_students,
    results_csv=NOV_BIM_RUNS_30_CSV,
    stats_csv=NOV_BIM_STATS_30_CSV,
    extra_kwargs_func=bim_nov_kwargs,
    n_runs=30,
)


[BIM-Nov] diversity + transferability + 30-run eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/results/nov_bim_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/results/nov_bim_prediction_diversity.csv

[BIM_NOV] Transfer matrix for eps=0.1...
  atk=student_05_enc_attn_bim_adv - def=student_05_enc_attn_bim_adv | eps=0.10 | RMSE=0.14109
  atk=student_05_enc_attn_bim_adv - def=student_06_ffn_bim_adv | eps=0.10 | RMSE=0.14088
  atk=student_05_enc_attn_bim_adv - def=student_07_norms_bim_adv | eps=0.10 | RMSE=0.14376
  atk=student_05_enc_attn_bim_adv - def=student_08_inproj_head_bim_adv | eps=0.10 | RMSE=0.13957
  atk=student_06_ffn_bim_adv - def=student_05_enc_attn_bim_adv | eps=0.10 | RMSE=0.14109
  atk=student_06_ffn_bim_adv - def=student_06_ffn_bim_adv | eps=0.10 | RMSE=0.14089
  atk=student_06_ffn_bim_adv - def=student_07_norms_bim_adv | eps=0.10 | RMSE=0.14376
  atk=student_06_ffn_bim_adv - def=

In [ ]:
print("\n[PGD-Nov] diversity + transferability + 30-run eval")

def pgd_nov_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / PGD_ITERS,
        "iters": PGD_ITERS,
        "random_start": True,
    }

_ = compute_weight_diversity(
    pgd_nov_students,
    pgd_nov_student_names,
    NOV_PGD_WEIGHT_DIVERSITY,
)
_ = compute_prediction_diversity(
    pgd_nov_students,
    test_dl,
    NOV_PGD_PRED_DIVERSITY,
)

_ = compute_transfer_matrix(
    students=pgd_nov_students,
    names=pgd_nov_student_names,
    loader=test_dl,
    attack_fn=pgd_attack,
    eps_list=EPS_LIST,
    extra_kwargs_func=pgd_nov_kwargs,
    out_csv=NOV_PGD_XFER_MATRIX_CSV,
    attack_label="pgd_nov",
)

df_pgd_nov_runs, pgd_nov_stats = run_morphence_eval(
    attack_name="pgd_nov",
    attack_fn=pgd_attack,
    eps_list=EPS_LIST,
    students=pgd_nov_students,
    results_csv=NOV_PGD_RUNS_30_CSV,
    stats_csv=NOV_PGD_STATS_30_CSV,
    extra_kwargs_func=pgd_nov_kwargs,
    n_runs=30,
)


[PGD-Nov] diversity + transferability + 30-run eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd/results/nov_pgd_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd/results/nov_pgd_prediction_diversity.csv

[PGD_NOV] Transfer matrix for eps=0.1...
  atk=student_05_enc_attn_pgd_adv - def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.11961
  atk=student_05_enc_attn_pgd_adv - def=student_06_ffn_pgd_adv | eps=0.10 | RMSE=0.08710
  atk=student_05_enc_attn_pgd_adv - def=student_07_norms_pgd_adv | eps=0.10 | RMSE=0.10071
  atk=student_05_enc_attn_pgd_adv - def=student_08_inproj_head_pgd_adv | eps=0.10 | RMSE=0.09186
  atk=student_06_ffn_pgd_adv - def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.08695
  atk=student_06_ffn_pgd_adv - def=student_06_ffn_pgd_adv | eps=0.10 | RMSE=0.10892
  atk=student_06_ffn_pgd_adv - def=student_07_norms_pgd_adv | eps=0.10 | RMSE=0.10226
  atk=student_06_ffn_pgd_adv - def=

In [ ]:
print("\n[FGSM-Nov] runs     :", NOV_FGSM_RUNS_30_CSV)
print("[BIM-Nov]  runs     :", NOV_BIM_RUNS_30_CSV)
print("[PGD-Nov]  runs     :", NOV_PGD_RUNS_30_CSV)
print("[FGSM-Nov] weight diversity:", NOV_FGSM_WEIGHT_DIVERSITY)
print("[BIM-Nov]  weight diversity:", NOV_BIM_WEIGHT_DIVERSITY)
print("[PGD-Nov]  weight diversity:", NOV_PGD_WEIGHT_DIVERSITY)
print("[FGSM-Nov] transfer matrix :", NOV_FGSM_XFER_MATRIX_CSV)
print("[BIM-Nov]  transfer matrix :", NOV_BIM_XFER_MATRIX_CSV)
print("[PGD-Nov]  transfer matrix :", NOV_PGD_XFER_MATRIX_CSV)


[FGSM-Nov] runs     : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/results/nov_fgsm_morphence_runs_30.csv
[BIM-Nov]  runs     : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/results/nov_bim_morphence_runs_30.csv
[PGD-Nov]  runs     : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd/results/nov_pgd_morphence_runs_30.csv
[FGSM-Nov] weight diversity: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/results/nov_fgsm_weight_diversity.csv
[BIM-Nov]  weight diversity: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/results/nov_bim_weight_diversity.csv
[PGD-Nov]  weight diversity: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd/results/nov_pgd_weight_diversity.csv
[FGSM-Nov] transfer matrix : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/results/nov_fgsm_transfer_matrix.csv
[BIM-Nov]  transfer matrix : /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/results/nov_bim_transfer_matrix.csv
[PGD-Nov]  transfer matrix : /content/drive/MyDrive/298/Jena/proj_v2/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

# recreate NOV directory + file paths

BASE_PROJ_DIR = "/content/drive/MyDrive/298/Jena/proj_v2"

JENANOV_DIR   = os.path.join(BASE_PROJ_DIR, "jenaNov")

FGSM_NOV_DIR         = os.path.join(JENANOV_DIR, "fgsm")
FGSM_NOV_RESULTS_DIR = os.path.join(FGSM_NOV_DIR, "results")

BIM_NOV_DIR          = os.path.join(JENANOV_DIR, "bim")
BIM_NOV_RESULTS_DIR  = os.path.join(BIM_NOV_DIR, "results")

PGD_NOV_DIR          = os.path.join(JENANOV_DIR, "pgd")
PGD_NOV_RESULTS_DIR  = os.path.join(PGD_NOV_DIR, "results")

NOV_FGSM_RUNS_30_CSV = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_runs_30.csv")
NOV_BIM_RUNS_30_CSV  = os.path.join(BIM_NOV_RESULTS_DIR,  "nov_bim_morphence_runs_30.csv")
NOV_PGD_RUNS_30_CSV  = os.path.join(PGD_NOV_RESULTS_DIR,  "nov_pgd_morphence_runs_30.csv")

print("FGSM NOV runs CSV:", NOV_FGSM_RUNS_30_CSV)
print("BIM  NOV runs CSV:", NOV_BIM_RUNS_30_CSV)
print("PGD  NOV runs CSV:", NOV_PGD_RUNS_30_CSV)

# config for NOV attacks
ATTACK_CONFIGS_NOV = [
    dict(
        name="FGSM-Nov",
        prefix="nov_fgsm",
        attack_key="rfgsm",   # column value in nov_fgsm_morphence_runs_30.csv
        runs_csv=NOV_FGSM_RUNS_30_CSV,
        results_dir=FGSM_NOV_RESULTS_DIR,
    ),
    dict(
        name="BIM-Nov",
        prefix="nov_bim",
        attack_key="bim_nov", # column value in nov_bim_morphence_runs_30.csv
        runs_csv=NOV_BIM_RUNS_30_CSV,
        results_dir=BIM_NOV_RESULTS_DIR,
    ),
    dict(
        name="PGD-Nov",
        prefix="nov_pgd",
        attack_key="pgd_nov", # column value in nov_pgd_morphence_runs_30.csv
        runs_csv=NOV_PGD_RUNS_30_CSV,
        results_dir=PGD_NOV_RESULTS_DIR,
    ),
]

# compute stats for one config
def compute_stats_for_attack_nov(cfg):
    name        = cfg["name"]
    prefix      = cfg["prefix"]
    attack_key  = cfg["attack_key"]
    runs_csv    = cfg["runs_csv"]
    results_dir = cfg["results_dir"]

    os.makedirs(results_dir, exist_ok=True)

    print("\n==============================================")
    print(f"Processing {name} runs")
    print(f"Runs CSV: {runs_csv}")
    print("==============================================")

    # load runs CSV
    df_runs = pd.read_csv(runs_csv)
    print("Loaded runs:", df_runs.shape)
    print(df_runs.head())

    # clean stats (base vs morph_ensemble)
    df_clean = df_runs[df_runs["attack"] == "clean"].copy()

    df_clean["model"] = pd.Categorical(
        df_clean["model"],
        categories=["base", "morph_ensemble"],
        ordered=True,
    )

    clean_stats = (
        df_clean
        .groupby("model", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("model")
    )

    print(f"\n[{name} | Clean] stats:")
    print(clean_stats.to_string(index=False))

    clean_csv = os.path.join(results_dir, f"{prefix}_clean_stats.csv")
    clean_stats.to_csv(clean_csv, index=False)
    print("Saved clean stats to:", clean_csv)

    # attack stats (base vs morph_rand)
    df_attack = df_runs[df_runs["attack"] == attack_key].copy()

    def model_attack_stats(df, model_name):
        st = (
            df[df["model"] == model_name]
            .groupby("epsilon", observed=False)["RMSE"]
            .agg(
                mean   ="mean",
                std    ="std",
                min    ="min",
                q1     =lambda s: s.quantile(0.25),
                median ="median",
                q3     =lambda s: s.quantile(0.75),
                max    ="max",
                n      ="count",
            )
            .reset_index()
            .sort_values("epsilon")
        )

        print(f"\n[{name} | {attack_key}] {model_name} stats:")
        print(st.to_string(index=False))

        out_csv = os.path.join(
            results_dir, f"{prefix}_{attack_key}_{model_name}_stats.csv"
        )
        st.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}")

    model_attack_stats(df_attack, "base")
    model_attack_stats(df_attack, "morph_rand")

# run for FGSM-Nov, BIM-Nov, PGD-Nov
for cfg in ATTACK_CONFIGS_NOV:
    compute_stats_for_attack_nov(cfg)


Mounted at /content/drive
FGSM NOV runs CSV: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/results/nov_fgsm_morphence_runs_30.csv
BIM  NOV runs CSV: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/bim/results/nov_bim_morphence_runs_30.csv
PGD  NOV runs CSV: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/pgd/results/nov_pgd_morphence_runs_30.csv

Processing FGSM-Nov runs
Runs CSV: /content/drive/MyDrive/298/Jena/proj_v2/jenaNov/fgsm/results/nov_fgsm_morphence_runs_30.csv
Loaded runs: (300, 6)
  attack  epsilon           model      RMSE  run  seed
0  clean      NaN            base  0.014594    1  3001
1  clean      NaN  morph_ensemble  0.015650    1  3001
2  rfgsm      0.1            base  0.134838    1  3001
3  rfgsm      0.1      morph_rand  0.107495    1  3001
4  rfgsm      0.2            base  0.219366    1  3001

[FGSM-Nov | Clean] stats:
         model     mean  std      min       q1   median       q3      max  n
          base 0.014594  0.0 0.014594 0.014594 0.014594 0.014